# 06 Personal Chef
In this workbook, we'll put all the techiques we have learnt so far to test to create a personal check that takes all the left-over ingredients in your fridge, searches the web for recipies that use those ingredients and then answer any follow up questions you may have.

In [2]:
from dotenv import load_dotenv

load_dotenv(override=True)

True

Let's add the web-search tool we created in the [using tools notebook](03_using_tools.ipynb)

In [3]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()


@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information
    Args:
        query (str): the search query
    Returns:
        Dict[str, Any]: the search results
    """
    print(f"------ Calling web_search({query}) tool ------")
    return tavily_client.search(query)

In [4]:
system_prompt = """
You ar an accomplished personal chef. The user will give you a set of ingredients they have 
left over in their house. Using the web_search tool, search the web for recipies using the
ingredients they have.
Return the recipe suggestions and eventually the recipe instructions if requested.
All recipe instruction must be returned as markdown step-by-step bulleted points. Assume that
the user is new to cooking, so instructions should be as detailed as possible.
"""

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="openai:gpt-5-nano",
    system_prompt=system_prompt,
    tools=[web_search],
    checkpointer=InMemorySaver(),
)

Let's start simple. Let us provide a list of ingredients to the agent & see what recipies I get back.

In [6]:
from langchain_core.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

message = HumanMessage(
    content="""I have some leftover chicken, rice, and cauliflower and capsicum. 
    What can I make for dinner?"""
)

response = agent.invoke({"messages": [message]}, config=config)
print(response["messages"][-1].content)

------ Calling web_search(one pan chicken cauliflower rice bell pepper recipe) tool ------
------ Calling web_search(chicken fried rice with cauliflower and peppers recipe) tool ------
------ Calling web_search(roasted chicken with cauliflower bell pepper rice recipe) tool ------
Nice! You’ve got the makings for several tasty dinners. Here are recipe ideas that use chicken, cauliflower, peppers (capsicum), and rice—plus a couple that work well with or without your leftover rice.

- Tex-Mex Cauliflower Rice with Chicken and Peppers (Smith Brothers Farms)
  - Quick, one-pan style. Sauté chicken with peppers, toss in cauliflower rice and Tex-Mex seasonings for a flavorful bowl.
  - Source: https://www.smithbrothersfarms.com/tex-mex-cauliflower-rice-with-chicken-and-peppers

- Cauliflower Fried Rice with Chicken and Peppers
  - Enjoy a fried-rice vibe without any actual rice. Use cauliflower “rice” with chicken, peppers, eggs, and soy-based seasonings.
  - Source: https://shuangyskitchensi

In [7]:
# now let's ask for step-by-step instructions on how to make one of the dishes above
message = HumanMessage(
    content="""Get me the Cauliflower Fried Rice with Chicken and Peppers recipe"""
)

response = agent.invoke({"messages": [message]}, config=config)
print(response["messages"][-1].content)

------ Calling web_search(Cauliflower Fried Rice With Chicken and Peppers Shuangy's Kitchensink) tool ------
Here’s a detailed, beginner-friendly recipe for Cauliflower Fried Rice with Chicken and Peppers, inspired by Shuangy’s Kitchensink.

Ingredients (serves 2-3)
- 1/2 lb (about 225 g) chicken thighs, boneless and skinless, cut into 1/2-inch cubes
- 1/2 head cauliflower, cut into florets and processed in a blender/food processor until it resembles rice (about 3 cups)
- 1/3 cup red bell pepper, diced
- 1/3 cup yellow or green bell pepper, diced (optional for color)
- 2 large eggs
- 2 cloves garlic, minced
- 1 teaspoon fresh ginger, grated (or 1/2 teaspoon ground ginger)
- 2–3 tablespoons soy sauce (or tamari for gluten-free)
- 1 teaspoon sesame oil (optional)
- 1–2 tablespoons neutral oil (like vegetable or canola) for cooking
- Optional garnishes: sliced green onions, a pinch of chili flakes or a drizzle of hot sauce

Equipment
- Large skillet or wok
- Knife and cutting board
- Food

Beautiful! Now let's get more realistic. 

### Adding an image - making our chef multi-modal
In this section, we'll ask the agent to figure out from an uploaded image what ingredients we have in our fridge and suggest recipies from there. This would mimic something like taking a photograph of your fridge on your mobile & asking Gemini for a recipe.

We'll leverage techiques we learnt in [multimodal inputs notebook](05_multimodal_inputs.ipynb)

In [8]:
from ipywidgets import FileUpload
from IPython.display import display

# upload single (multiple=False) PNG (accept=".png") files only!
uploader = FileUpload(accept=".jpg", multiple=False)
display(uploader)

FileUpload(value=(), accept='.jpg', description='Upload')

In [9]:
# let's encode the image using base64 encoding
import base64

uploaded_file = uploader.value[0]
encoded_image = base64.b64encode(bytes(uploaded_file["content"])).decode("utf-8")

In [10]:
# as a first step, let's ask our agent what ingredients it can pick out from the image.

from langchain_core.messages import HumanMessage


config2 = {"configurable": {"thread_id": "thread_2"}}

ingredients_message = """Look at the image of my fridge, with leftover ingredients. 
Let me know what ingredients you can identify. 
Return a markdown bulleted list of the just ingredients you can identify in the image 
and no other text!"""

multimodal_question = HumanMessage(
    content=[
        {"type": "text", "text": ingredients_message},
        {"type": "image", "base64": encoded_image, "mime_type": "image/jpeg"},
    ]
)

response = agent.invoke({"messages": [multimodal_question]}, config=config2)
# capture the ingredients (as we'll need them in the next call to agent)
ingredients_found = response["messages"][-1].content
print(ingredients_found)

- Cheese (assorted blocks)
- Pickles (jar)
- Parsley
- Dill
- Lettuce
- Tomatoes
- Cucumber
- Ham slices


In [11]:
# now let's ask the agent to suggest recipies using ingredients it found above
recipies_message = HumanMessage(
    content=f"""I have the following leftover ingredients in my fridge:
    {ingredients_found} 
    What can I make for dinner? 
    """
)

response = agent.invoke({"messages": [recipies_message]}, config=config2)
print(response["messages"][-1].content)

------ Calling web_search(ham cheese lettuce tomato cucumber wrap recipe) tool ------
------ Calling web_search(ham and cheese omelette dill parsley recipe) tool ------
------ Calling web_search(cheese pickles sandwich recipe) tool ------
------ Calling web_search(cold cut platter with pickles tomatoes cucumber dill parsley recipe) tool ------
------ Calling web_search(vegetable salad with ham cheese cucumber tomato dill parsley recipe) tool ------
- Lettuce-Wrapped Ham, Cheese & Veggie Rolls
- Ham, Cheese & Pickle Sandwiches or Wraps with Tomato & Cucumber
- Hearty Ham, Tomato & Cucumber Salad with Parsley & Dill
- Quick Ham & Cheese Roll-Ups with Pickles
- Cheese & Ham Platter with Tomato & Cucumber Slices (simple dinner plate)


In [12]:
recipe_message = HumanMessage(
    content="""Get me the 'Hearty Ham, Tomato & Cucumber Salad with Parsley & Dill' recipe"""
)

response = agent.invoke({"messages": [recipe_message]}, config=config2)
print(response["messages"][-1].content)

------ Calling web_search(Hearty Ham Tomato Cucumber Salad Parsley Dill recipe) tool ------
- Hearty Ham, Tomato & Cucumber Salad with Parsley & Dill
  - Ingredients
    - 200–250 g ham, diced
    - 2 cups cherry tomatoes or 2–3 medium tomatoes, chopped
    - 1 large cucumber, diced or sliced
    - 1/4 cup fresh parsley, chopped
    - 1 tablespoon fresh dill, chopped
    - 2 tablespoons olive oil
    - 1 tablespoon lemon juice or white wine vinegar
    - Salt and pepper to taste
    - Optional: 1/4 red onion, thinly sliced
  - Instructions
    - Step 1: In a large bowl, combine ham, tomatoes, cucumber, parsley, dill, and red onion if using.
    - Step 2: In a small bowl, whisk olive oil with lemon juice (or vinegar), salt, and pepper.
    - Step 3: Drizzle the dressing over the salad and toss gently to coat all ingredients.
    - Step 4: Let the salad rest for 5–10 minutes to allow flavors to meld. Taste and adjust seasoning.
    - Step 5: Serve as a light main or side, optionally on a

Wow! That was a really cool project.

Go on and make the recipe above & let me know how good it tastes 🤤